<a href="https://colab.research.google.com/github/sathundorn/Super-AI-Engineer-Season-6/blob/main/Hackaton2_601402.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -y
!apt-get install -y tesseract-ocr tesseract-ocr-tha
!pip install -q pandas numpy pillow easyocr typhoon-ocr

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,943 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,842 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,914 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,618 kB]
Get:14 http://sec

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from __future__ import annotations

import argparse
import json
import math
import os
import re
import time
import subprocess
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

try:
    import easyocr
except Exception:
    easyocr = None

try:
    from typhoon_ocr import ocr_document as typhoon_ocr_document
except Exception:
    typhoon_ocr_document = None


PARTY_LIST_X = (1500, 1835)
CONSTITUENCY_X = (1680, 1985)
ROW_DARK_THRESHOLD = 175
ROW_PROJ_THRESHOLD = 0.075
COL_PROJ_THRESHOLD = 0.02

THAI_DIGITS = str.maketrans("0123456789", "๐๑๒๓๔๕๖๗๘๙")


@dataclass
class RowGroup:
    y1: int
    y2: int
    peak: float


def load_gray(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))


def cluster_positions(values: np.ndarray, max_gap: int = 2) -> list[tuple[int, int]]:
    if len(values) == 0:
        return []
    values = np.sort(values)
    start = prev = int(values[0])
    groups: list[tuple[int, int]] = []
    for val in values[1:]:
        val = int(val)
        if val <= prev + max_gap:
            prev = val
            continue
        groups.append((start, prev))
        start = prev = val
    groups.append((start, prev))
    return groups


def row_groups(gray: np.ndarray, x1: int, x2: int) -> list[RowGroup]:
    mask = gray[:, x1:x2] < ROW_DARK_THRESHOLD
    proj = mask.mean(axis=1)
    rows = np.where(proj > ROW_PROJ_THRESHOLD)[0]
    groups = []
    for y1, y2 in cluster_positions(rows, max_gap=6):
        groups.append(RowGroup(y1, y2, float(proj[y1 : y2 + 1].max())))
    return groups


def non_line_groups(groups: Iterable[RowGroup], min_y: int = 0) -> list[RowGroup]:
    kept = []
    for group in groups:
        height = group.y2 - group.y1 + 1
        if group.y1 < min_y:
            continue
        if group.peak >= 0.95 and height <= 8:
            continue
        if height < 6:
            continue
        kept.append(group)
    return kept


def is_table_page(gray: np.ndarray, doc_type: str) -> bool:
    x1, x2 = PARTY_LIST_X if doc_type == "party_list" else CONSTITUENCY_X
    groups = non_line_groups(row_groups(gray, x1, x2), min_y=450)
    return len(groups) >= 4


def page_paths(images_dir: Path, doc_id: str) -> list[Path]:
    paths = []
    base = images_dir / f"{doc_id}.png"
    if base.exists():
        paths.append(base)
    page = 2
    while True:
        path = images_dir / f"{doc_id}_page{page}.png"
        if not path.exists():
            break
        paths.append(path)
        page += 1
    return paths


def extract_row_crops(doc_id: str, images_dir: Path, target_rows: int) -> list[np.ndarray]:
    doc_type = "party_list" if doc_id.startswith("party_list") else "constituency"
    x1, x2 = PARTY_LIST_X if doc_type == "party_list" else CONSTITUENCY_X
    crops: list[np.ndarray] = []
    all_paths = page_paths(images_dir, doc_id)
    for path in all_paths[1:]:
        gray = load_gray(path)
        if not is_table_page(gray, doc_type):
            continue
        all_groups = row_groups(gray, x1, x2)
        line_groups = [
            group
            for group in all_groups
            if group.y1 >= 450 and group.peak >= 0.95 and (group.y2 - group.y1 + 1) <= 14
        ]
        if line_groups:
            if doc_type == "party_list" and len(line_groups) > 1:
                start_y = line_groups[1].y2 + 1
            else:
                start_y = line_groups[0].y2 + 1
        else:
            start_y = 450
        groups = non_line_groups(all_groups, min_y=start_y)
        remaining = target_rows - len(crops)
        if remaining <= 0:
            break
        for group in groups[:remaining]:
            y1 = max(group.y1 - 20, 0)
            y2 = min(group.y2 + 20, gray.shape[0] - 1)
            crop = gray[y1 : y2 + 1, x1:x2]
            crops.append(crop)
        if len(crops) >= target_rows:
            break
    return crops


def trim_binary(mask: np.ndarray, pad: int = 2) -> np.ndarray:
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return mask
    y1, y2 = max(int(ys.min()) - pad, 0), min(int(ys.max()) + pad + 1, mask.shape[0])
    x1, x2 = max(int(xs.min()) - pad, 0), min(int(xs.max()) + pad + 1, mask.shape[1])
    return mask[y1:y2, x1:x2]


def segment_char_masks(row_crop: np.ndarray) -> list[np.ndarray]:
    crop = Image.fromarray(row_crop)
    crop = ImageOps.autocontrast(crop)
    arr = np.array(crop)
    mask = arr < ROW_DARK_THRESHOLD
    col_proj = mask.mean(axis=0)
    cols = np.where(col_proj > COL_PROJ_THRESHOLD)[0]
    char_masks: list[np.ndarray] = []
    for x1, x2 in cluster_positions(cols, max_gap=1):
        char = mask[:, x1 : x2 + 1]
        if char.sum() < 10:
            continue
        char = trim_binary(char, pad=1)
        h, w = char.shape
        if h < 8 or w < 4:
            continue
        char_masks.append(char)
    return char_masks


def normalize_char(mask: np.ndarray, size: int = 28) -> np.ndarray:
    img = Image.fromarray((~mask).astype(np.uint8) * 255)
    img = img.resize((size - 4, size - 4))
    canvas = Image.new("L", (size, size), color=255)
    canvas.paste(img, (2, 2))
    arr = 1.0 - (np.array(canvas).astype(np.float32) / 255.0)
    return arr


def number_labels(votes: int, doc_type: str) -> list[str]:
    _ = doc_type
    return list(str(votes))


class TemplateMatcher:
    def __init__(self) -> None:
        self.templates: dict[str, list[np.ndarray]] = defaultdict(list)

    def add(self, label: str, mask: np.ndarray) -> None:
        self.templates[label].append(normalize_char(mask))

    def predict_one(self, mask: np.ndarray) -> tuple[str, float]:
        query = normalize_char(mask)
        best_label = ""
        best_score = math.inf
        for label, templates in self.templates.items():
            for template in templates:
                score = float(np.mean(np.abs(query - template)))
                if score < best_score:
                    best_score = score
                    best_label = label
        return best_label, best_score

    def predict_number(self, row_crop: np.ndarray) -> tuple[str, list[float]]:
        chars = segment_char_masks(row_crop)
        labels = []
        scores = []
        for char in chars:
            label, score = self.predict_one(char)
            labels.append(label)
            scores.append(score)
        value = "".join(ch for ch in labels if ch.isdigit())
        return value, scores


THAI_TO_ARABIC = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
THAI_NUM_DIGITS = {
    "ศูนย์": 0,
    "หนึ่ง": 1,
    "เอ็ด": 1,
    "สอง": 2,
    "ยี่": 2,
    "สาม": 3,
    "สี่": 4,
    "ห้า": 5,
    "หก": 6,
    "เจ็ด": 7,
    "แปด": 8,
    "เก้า": 9,
}
THAI_NUM_UNITS = {
    "สิบ": 10,
    "ร้อย": 100,
    "พัน": 1000,
    "หมื่น": 10000,
    "แสน": 100000,
    "ล้าน": 1000000,
}
THAI_NUM_VOCAB = sorted(list(THAI_NUM_DIGITS) + list(THAI_NUM_UNITS), key=len, reverse=True)
EASYOCR_READER = None


def ocr_numeric_text(row_crop: np.ndarray, work_dir: Path, idx: str = "row") -> str:
    work_dir.mkdir(parents=True, exist_ok=True)
    safe_idx = re.sub(r"[^A-Za-z0-9_.-]+", "_", idx)
    tmp_path = work_dir / f"ocr_row_{safe_idx}.png"
    Image.fromarray(row_crop).save(tmp_path)
    out = subprocess.check_output(
        [
            "tesseract",
            str(tmp_path),
            "stdout",
            "-l",
            "tha+eng",
            "--psm",
            "7",
            "tsv",
        ],
        text=True,
    )
    numeric_parts: list[str] = []
    for line in out.splitlines()[1:]:
        if "\t" not in line:
            continue
        text = line.split("\t")[-1].strip()
        if not text:
            continue
        if "(" in text:
            break
        pieces = re.findall(r"[0-9๐-๙]+", text)
        if pieces:
            numeric_parts.extend(pieces)
    value = "".join(numeric_parts).translate(THAI_TO_ARABIC).replace(",", "")
    return re.sub(r"\D", "", value)


def get_easyocr_reader(work_dir: Path):
    global EASYOCR_READER
    if EASYOCR_READER is None and easyocr is not None:
        models_dir = work_dir.parent / ".easyocr_models"
        user_dir = work_dir.parent / ".easyocr_user"
        if (models_dir / "craft_mlt_25k.pth").exists() and (models_dir / "thai.pth").exists():
            EASYOCR_READER = easyocr.Reader(
                ["th", "en"],
                gpu=False,
                model_storage_directory=str(models_dir),
                user_network_directory=str(user_dir),
                download_enabled=False,
                verbose=False,
            )
    return EASYOCR_READER


def parse_thai_number_text(text: str) -> int | None:
    text = text.replace(" ", "")
    replacements = {
        "หนึง": "หนึ่ง",
        "หนืง": "หนึ่ง",
        "หมน": "หมื่น",
        "หมึน": "หมื่น",
        "หมีน": "หมื่น",
        "หม่น": "หมื่น",
        "เสี่": "สี่",
        "สี": "สี่",
        "เกธ": "เก้า",
        "เจด": "เจ็ด",
        "สิอ": "สิบ",
        "สออ": "สอง",
        "ยี": "ยี่",
        "ยิ": "ยี่",
        "พ้": "พัน",
        "พั": "พัน",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)

    tokens: list[str] = []
    i = 0
    while i < len(text):
        for token in THAI_NUM_VOCAB:
            if text.startswith(token, i):
                tokens.append(token)
                i += len(token)
                break
        else:
            i += 1

    if not tokens:
        return None

    total = 0
    current = 0
    digit = None
    for token in tokens:
        if token == "ล้าน":
            total = (total + current) * 1000000
            current = 0
            digit = None
        elif token in THAI_NUM_DIGITS:
            digit = THAI_NUM_DIGITS[token]
        else:
            unit = THAI_NUM_UNITS[token]
            value = 1 if digit is None else digit
            current += value * unit
            digit = None
    if digit is not None:
        current += digit
    return total + current


def ocr_party_list_text(row_crop: np.ndarray, work_dir: Path, idx: str) -> str:
    reader = get_easyocr_reader(work_dir)
    if reader is None:
        return ocr_numeric_text(row_crop, work_dir, idx)
    tmp_path = work_dir / f"easy_{idx}.png"
    Image.fromarray(row_crop).save(tmp_path)
    text = " ".join(reader.readtext(str(tmp_path), detail=0, paragraph=False))
    digits = "".join(re.findall(r"[0-9๐-๙,]+", text)).translate(THAI_TO_ARABIC).replace(",", "")
    digits = re.sub(r"\D", "", digits)
    thai_value = None
    if "(" in text:
        thai_value = parse_thai_number_text(text.split("(", 1)[1])
    if digits and thai_value is not None:
        if abs(len(digits) - len(str(thai_value))) >= 2 or digits.startswith("0"):
            return str(thai_value)
        return digits
    if digits:
        return digits
    if thai_value is not None:
        return str(thai_value)
    return "0"


def extract_table_votes(markdown: str) -> list[str]:
    votes: list[str] = []
    for match in re.finditer(r"<tr>(.*?)<\/tr>", markdown, re.S):
        tds = re.findall(r"<td>(.*?)<\/td>", match.group(1), re.S)
        if len(tds) < 3:
            continue
        cols = [re.sub(r"<.*?>", "", td).strip().translate(THAI_TO_ARABIC) for td in tds[:3]]
        vote_match = re.search(r"([0-9][0-9,]*)", cols[2])
        if not vote_match:
            continue
        votes.append(vote_match.group(1).replace(",", ""))
    return votes


def extract_good_ballots(markdown: str) -> int | None:
    text = markdown.translate(THAI_TO_ARABIC)
    match = re.search(r"4\\.1\\s*บัตรดี\\s*([0-9][0-9,]*)", text)
    if not match:
        match = re.search(r"๔\\.๑\\s*บัตรดี\\s*([0-9][0-9,]*)", text)
    if not match:
        return None
    return int(match.group(1).replace(",", ""))


def extract_constituency_votes(markdown: str) -> list[str]:
    votes: list[str] = []
    good_ballots = extract_good_ballots(markdown)
    for match in re.finditer(r"<tr>(.*?)<\/tr>", markdown, re.S):
        tds = re.findall(r"<td(?:\\s+colspan=\"\\d+\")?>(.*?)<\/td>", match.group(1), re.S)
        if len(tds) < 4:
            continue
        cols = [re.sub(r"<.*?>", "", td).strip().translate(THAI_TO_ARABIC) for td in tds[:4]]
        vote_match = re.search(r"([0-9][0-9,]*)", cols[3])
        if "รวมคะแนน" in cols[0] or "รวมคะแนน" in cols[1] or "รวมคะแนน" in cols[2] or not vote_match:
            continue
        vote = vote_match.group(1).replace(",", "")
        vote_int = int(vote) if vote.isdigit() else None
        # Typhoon occasionally leaks the total "valid ballots" into the last candidate row.
        if good_ballots is not None and vote_int is not None and vote_int >= int(good_ballots * 0.9):
            continue
        votes.append(vote)
    return votes


def ocr_typhoon_page(path: Path, cache_dir: Path) -> str:
    if typhoon_ocr_document is None:
        raise RuntimeError("typhoon_ocr is not installed")
    api_key = os.getenv("TYPHOON_OCR_API_KEY") or os.getenv("TYPHOON_API_KEY")
    if not api_key:
        raise RuntimeError("TYPHOON_OCR_API_KEY is not set")

    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{path.name}.md"
    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8")

    last_error: Exception | None = None
    for delay in (0, 15, 45):
        if delay:
            time.sleep(delay)
        try:
            text = typhoon_ocr_document(str(path), task_type="v1.5", model="typhoon-ocr", api_key=api_key)
            cache_path.write_text(text, encoding="utf-8")
            time.sleep(3.5)
            return text
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Typhoon OCR failed for {path.name}: {last_error}")


def predict_party_list_typhoon(doc_id: str, images_dir: Path, n_rows: int, work_dir: Path) -> list[str] | None:
    if typhoon_ocr_document is None:
        return None
    if not (os.getenv("TYPHOON_OCR_API_KEY") or os.getenv("TYPHOON_API_KEY")):
        return None

    cache_dir = work_dir.parent / ".typhoon_cache"
    votes: list[str] = []
    for path in page_paths(images_dir, doc_id):
        try:
            text = ocr_typhoon_page(path, cache_dir)
        except Exception:
            continue
        votes.extend(extract_table_votes(text))

    if not votes:
        return None
    out = votes[:n_rows]
    while len(out) < n_rows:
        out.append("0")
    return out


def predict_constituency_typhoon(
    doc_id: str,
    n_rows: int,
    images_dir: Path,
    work_dir: Path,
) -> list[str] | None:
    if typhoon_ocr_document is None:
        return None
    if not (os.getenv("TYPHOON_OCR_API_KEY") or os.getenv("TYPHOON_API_KEY")):
        return None

    cache_dir = work_dir.parent / ".typhoon_cache"
    votes: list[str] = []
    for path in page_paths(images_dir, doc_id):
        try:
            text = ocr_typhoon_page(path, cache_dir)
        except Exception:
            continue
        votes.extend(extract_constituency_votes(text))

    if not votes:
        return None
    out = votes[:n_rows]
    while len(out) < n_rows:
        out.append("0")
    return out


def sample_doc_rows(sample_path: Path) -> list[int]:
    with open(sample_path, encoding="utf-8") as fh:
        obj = json.load(fh)
    return [int(row["votes"]) for row in obj["results"]]


def build_matcher(images_dir: Path, sample_dir: Path) -> TemplateMatcher:
    matcher = TemplateMatcher()
    for sample_path in sorted(sample_dir.glob("*.json")):
        doc_id = sample_path.stem
        doc_type = "party_list" if doc_id.startswith("party_list") else "constituency"
        votes = sample_doc_rows(sample_path)
        row_crops = extract_row_crops(doc_id, images_dir, len(votes))
        for row_crop, vote in zip(row_crops, votes):
            char_masks = segment_char_masks(row_crop)
            labels = number_labels(vote, doc_type)
            if len(char_masks) != len(labels):
                continue
            for label, mask in zip(labels, char_masks):
                matcher.add(label, mask)
    return matcher


def predict_doc_votes(doc_id: str, images_dir: Path, n_rows: int, work_dir: Path) -> list[str]:
    preds: list[str] = []
    doc_type = "party_list" if doc_id.startswith("party_list") else "constituency"
    if doc_type == "party_list":
        typhoon_preds = predict_party_list_typhoon(doc_id, images_dir, n_rows, work_dir)
        if typhoon_preds is not None:
            return typhoon_preds
    else:
        typhoon_preds = predict_constituency_typhoon(doc_id, n_rows, images_dir, work_dir)
        if typhoon_preds is not None:
            return typhoon_preds

    row_crops = extract_row_crops(doc_id, images_dir, n_rows)
    for i, crop in enumerate(row_crops[:n_rows]):
        if doc_type == "party_list":
            value = ocr_party_list_text(crop, work_dir, f"{doc_id}_{i}")
        else:
            value = ocr_numeric_text(crop, work_dir, f"{doc_id}_{i}")
        preds.append(value or "0")
    while len(preds) < n_rows:
        preds.append("0")
    return preds


def mean_levenshtein(a: list[str], b: list[str]) -> float:
    def dist(x: str, y: str) -> int:
        dp = list(range(len(y) + 1))
        for i, cx in enumerate(x, start=1):
            prev = dp[0]
            dp[0] = i
            for j, cy in enumerate(y, start=1):
                cur = dp[j]
                cost = 0 if cx == cy else 1
                dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
                prev = cur
        return dp[-1]

    return float(np.mean([dist(x, y) for x, y in zip(a, b)]))


def validate_samples(images_dir: Path, sample_dir: Path, work_dir: Path) -> None:
    rows = []
    for sample_path in sorted(sample_dir.glob("*.json")):
        doc_id = sample_path.stem
        with open(sample_path, encoding="utf-8") as fh:
            obj = json.load(fh)
        truth = [str(row["votes"]) for row in obj["results"]]
        pred = predict_doc_votes(doc_id, images_dir, len(truth), work_dir)
        rows.append(
            {
                "doc_id": doc_id,
                "rows": len(truth),
                "mean_levenshtein": mean_levenshtein(pred, truth),
                "exact": sum(p == t for p, t in zip(pred, truth)),
            }
        )
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print("\nmacro_mean_levenshtein =", df["mean_levenshtein"].mean())


def make_submission(
    template_csv: Path,
    images_dir: Path,
    sample_dir: Path,
    output_csv: Path,
    validate: bool,
) -> None:
    work_dir = output_csv.parent / ".ocr_tmp"
    work_dir.mkdir(parents=True, exist_ok=True)
    if validate:
        validate_samples(images_dir, sample_dir, work_dir)

    submission = pd.read_csv(template_csv, dtype={"votes": str})
    partial_csv = output_csv.with_name(f"{output_csv.stem}_partial.csv")
    groups = list(submission.groupby("doc_id", sort=False))
    out_votes: list[str] = []
    processed_docs = 0

    if partial_csv.exists():
        partial = pd.read_csv(partial_csv, dtype={"votes": str})
        out_votes = partial["votes"].astype(str).tolist()
        seen_rows = len(out_votes)
        row_total = 0
        for doc_id, group in groups:
            next_total = row_total + len(group)
            if next_total <= seen_rows:
                processed_docs += 1
                row_total = next_total
                continue
            break
        print(f"resuming from {processed_docs}/{len(groups)} docs and {seen_rows} rows")

    for doc_id, group in groups[processed_docs:]:
        preds = predict_doc_votes(doc_id, images_dir, len(group), work_dir)
        out_votes.extend(preds)
        processed_docs += 1
        if processed_docs % 5 == 0 or processed_docs == len(groups):
            partial = submission.iloc[: len(out_votes)][["id"]].copy()
            partial["votes"] = out_votes
            partial.to_csv(partial_csv, index=False)
            print(f"progress {processed_docs}/{len(groups)} docs -> {partial_csv}")
    submission["votes"] = out_votes
    submission[["id", "votes"]].to_csv(output_csv, index=False)
    print(f"wrote {output_csv}")


def main(argv=None) -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--images-dir", required=True, type=Path)
    parser.add_argument("--sample-dir", required=True, type=Path)
    parser.add_argument("--template-csv", required=True, type=Path)
    parser.add_argument("--output-csv", default=Path("submission.csv"), type=Path)
    parser.add_argument("--validate", action="store_true")
    args = parser.parse_args(argv)

    make_submission(
        template_csv=args.template_csv,
        images_dir=args.images_dir,
        sample_dir=args.sample_dir,
        output_csv=args.output_csv,
        validate=args.validate,
    )


In [ ]:
main([
  '--images-dir', '/content/drive/MyDrive/data/images',
  '--sample-dir', '/content/drive/MyDrive/data/sample_labels',
  '--template-csv', '/content/drive/MyDrive/data/submission_template_v3.csv',
  '--output-csv', '/content/submission_upload_v3.csv' # It's fine to leave output here, or move it to Drive
])

progress 5/300 docs -> /content/submission_upload_v3_partial.csv
progress 10/300 docs -> /content/submission_upload_v3_partial.csv
progress 15/300 docs -> /content/submission_upload_v3_partial.csv
progress 20/300 docs -> /content/submission_upload_v3_partial.csv
progress 25/300 docs -> /content/submission_upload_v3_partial.csv
progress 30/300 docs -> /content/submission_upload_v3_partial.csv
progress 35/300 docs -> /content/submission_upload_v3_partial.csv
progress 40/300 docs -> /content/submission_upload_v3_partial.csv
progress 45/300 docs -> /content/submission_upload_v3_partial.csv
progress 50/300 docs -> /content/submission_upload_v3_partial.csv
progress 55/300 docs -> /content/submission_upload_v3_partial.csv
progress 60/300 docs -> /content/submission_upload_v3_partial.csv
progress 65/300 docs -> /content/submission_upload_v3_partial.csv
progress 70/300 docs -> /content/submission_upload_v3_partial.csv
progress 75/300 docs -> /content/submission_upload_v3_partial.csv
progress 80